# Feature Engineering Pipeline

This notebook transforms the cleaned heart attack dataset into a model-ready feature set.

**Pipeline stages covered here:**
1. Load the cleaned data
2. Construct new domain-driven features (before encoding)
3. Encode categorical features:
   - **Ordinal Encoding** for columns with a natural order
   - **One-Hot Encoding** for nominal columns
4. Scale numerical features with **StandardScaler**
5. Final feature matrix overview
6. Save the processed dataset to `data/processed/`

The output of this notebook feeds directly into `05_model_training.ipynb`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

from sklearn.preprocessing import OrdinalEncoder, StandardScaler
import joblib

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

print("Libraries loaded successfully.")

## 1. Load Cleaned Data

In [ ]:
df = pd.read_csv('../data/interim/heart_attack_dataset_cleaned.csv')

print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head()

In [ ]:
# Drop patient_id — unique identifier with no predictive value
df.drop(columns=['patient_id'], inplace=True)

TARGET = 'heart_attack_risk'
print(f"Target class balance:\n{df[TARGET].value_counts()}")
print(f"\nClass imbalance ratio: {df[TARGET].value_counts(normalize=True).round(3).to_dict()}")

## 2. Feature Construction

New features are built **before** encoding so we can still use raw categorical string values in composite scores.

| Feature | Type | Rationale |
|---|---|---|
| `age_group` | Ordinal (engineered) | Age bins capture non-linear risk thresholds |
| `bp_category` | Ordinal (engineered) | AHA hypertension staging |
| `hr_age_ratio` | Numeric | Max HR as proportion of age-predicted max (stress test signal) |
| `heart_rate_reserve` | Numeric | Max HR minus 60 bpm resting baseline |
| `cholesterol_bp_ratio` | Numeric | Combined vascular load proxy |
| `lifestyle_risk_score` | Numeric | Smoking + alcohol + physical inactivity |
| `comorbidity_score` | Numeric | Diabetes + family history + high stress |

In [ ]:
# --- Age Group (ordinal bins) ---
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 40, 50, 60, 70, 120],
    labels=['<40', '40-49', '50-59', '60-69', '70+']
).astype(str)

# --- Blood Pressure Category (AHA stages, ordinal) ---
def bp_category(bp):
    if bp < 120:
        return 'Normal'
    elif bp < 130:
        return 'Elevated'
    elif bp < 140:
        return 'Stage1'
    else:
        return 'Stage2'

df['bp_category'] = df['resting_bp'].apply(bp_category)

# --- Heart Rate Age Ratio ---
df['hr_age_ratio'] = df['max_heart_rate'] / (220 - df['age'])

# --- Heart Rate Reserve ---
df['heart_rate_reserve'] = df['max_heart_rate'] - 60

# --- Cholesterol × BP Ratio ---
df['cholesterol_bp_ratio'] = df['cholesterol'] / (df['resting_bp'] + 1e-5)

# --- Lifestyle Risk Score (uses raw strings before ordinal encoding) ---
smoking_map  = {'Never': 0, 'Former': 1, 'Current': 2}
alcohol_map  = {'Non-drinker': 0, 'Moderate': 1, 'Heavy': 2}
activity_map = {'High': 0, 'Moderate': 1, 'Low': 2}

df['lifestyle_risk_score'] = (
    df['smoking_status'].map(smoking_map) +
    df['alcohol_consumption'].map(alcohol_map) +
    df['physical_activity'].map(activity_map)
)

# --- Comorbidity Score ---
df['comorbidity_score'] = (
    df['diabetes'] +
    df['family_history'] +
    (df['stress_level'] >= 7).astype(int)
)

new_feats = ['age_group', 'bp_category', 'hr_age_ratio', 'heart_rate_reserve',
             'cholesterol_bp_ratio', 'lifestyle_risk_score', 'comorbidity_score']
print("Engineered features summary:")
print(df[new_feats].describe(include='all').round(3))

### Distributions of Engineered Numeric Features

In [ ]:
# Distribution for hr_age_ratio
feat = 'hr_age_ratio'
plt.figure(figsize=(6, 4))
plt.hist(df[feat], bins=20, edgecolor='black', color='steelblue', alpha=0.8)
plt.title(f'Distribution of {feat}', fontsize=12, fontweight='bold')
plt.xlabel('Value')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution for heart_rate_reserve
feat = 'heart_rate_reserve'
plt.figure(figsize=(6, 4))
plt.hist(df[feat], bins=20, edgecolor='black', color='steelblue', alpha=0.8)
plt.title(f'Distribution of {feat}', fontsize=12, fontweight='bold')
plt.xlabel('Value')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution for cholesterol_bp_ratio
feat = 'cholesterol_bp_ratio'
plt.figure(figsize=(6, 4))
plt.hist(df[feat], bins=20, edgecolor='black', color='steelblue', alpha=0.8)
plt.title(f'Distribution of {feat}', fontsize=12, fontweight='bold')
plt.xlabel('Value')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution for lifestyle_risk_score
feat = 'lifestyle_risk_score'
plt.figure(figsize=(6, 4))
plt.hist(df[feat], bins=20, edgecolor='black', color='steelblue', alpha=0.8)
plt.title(f'Distribution of {feat}', fontsize=12, fontweight='bold')
plt.xlabel('Value')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution for comorbidity_score
feat = 'comorbidity_score'
plt.figure(figsize=(6, 4))
plt.hist(df[feat], bins=20, edgecolor='black', color='steelblue', alpha=0.8)
plt.title(f'Distribution of {feat}', fontsize=12, fontweight='bold')
plt.xlabel('Value')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 3. Encoding Categorical Features

Columns are split by whether they have a **natural order** or not:

| Column | Type | Encoding |
|---|---|---|
| `smoking_status` | Ordinal | `Never` → 0 · `Former` → 1 · `Current` → 2 |
| `alcohol_consumption` | Ordinal | `Non-drinker` → 0 · `Moderate` → 1 · `Heavy` → 2 |
| `physical_activity` | Ordinal | `Low` → 0 · `Moderate` → 1 · `High` → 2 |
| `st_slope` | Ordinal | `Down` → 0 · `Flat` → 1 · `Up` → 2 |
| `age_group` | Ordinal | `<40` → 0 · `40-49` → 1 · `50-59` → 2 · `60-69` → 3 · `70+` → 4 |
| `bp_category` | Ordinal | `Normal` → 0 · `Elevated` → 1 · `Stage1` → 2 · `Stage2` → 3 |
| `gender` | Nominal | One-Hot Encoded |
| `chest_pain_type` | Nominal | One-Hot Encoded |
| `resting_ecg` | Nominal | One-Hot Encoded |
| `thalassemia` | Nominal | One-Hot Encoded |

### 3.1 Ordinal Encoding

In [ ]:
# Define explicit category orders for each ordinal column
ordinal_config = {
    'smoking_status'     : ['Never', 'Former', 'Current'],
    'alcohol_consumption': ['Non-drinker', 'Moderate', 'Heavy'],
    'physical_activity'  : ['Low', 'Moderate', 'High'],
    'st_slope'           : ['Down', 'Flat', 'Up'],
    'age_group'          : ['<40', '40-49', '50-59', '60-69', '70+'],
    'bp_category'        : ['Normal', 'Elevated', 'Stage1', 'Stage2'],
}

ordinal_cols      = list(ordinal_config.keys())
ordinal_categories = [ordinal_config[c] for c in ordinal_cols]

ordinal_encoder = OrdinalEncoder(
    categories=ordinal_categories,
    dtype=int
)

df[ordinal_cols] = ordinal_encoder.fit_transform(df[ordinal_cols])

# Persist encoder for inference
os.makedirs('../models', exist_ok=True)
joblib.dump(ordinal_encoder, '../models/ordinal_encoder.pkl')

print("Ordinal encoding applied:")
for col, cats in zip(ordinal_cols, ordinal_categories):
    mapping = {cat: i for i, cat in enumerate(cats)}
    print(f"  {col:25s}: {mapping}")

print("\nOrdinal encoder saved to '../models/ordinal_encoder.pkl'")
df[ordinal_cols].head()

### 3.2 One-Hot Encoding

In [ ]:
nominal_cols = ['gender', 'chest_pain_type', 'resting_ecg', 'thalassemia']

print(f"Nominal columns and their unique values:")
for c in nominal_cols:
    print(f"  {c}: {sorted(df[c].unique().tolist())}")

In [ ]:
shape_before = df.shape

df = pd.get_dummies(df, columns=nominal_cols, drop_first=True, dtype=int)

print(f"Shape before OHE : {shape_before}")
print(f"Shape after OHE  : {df.shape}")
print(f"\nNew OHE columns:")
for c in df.columns:
    if any(nom in c for nom in nominal_cols):
        print(f"  {c}")

In [ ]:
# Verify no object/category columns remain
remaining = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Remaining non-numeric columns: {remaining if remaining else 'None — all encoded ✓'}")
df.dtypes

## 4. StandardScaler — Scale Numerical Features

**StandardScaler** (zero-mean, unit-variance) is applied to all continuous numerical columns.  
Binary flags, ordinal-encoded, and OHE columns are intentionally left unscaled.

Fitted scaler is saved to `models/standard_scaler.pkl` for inference.

In [ ]:
scale_cols = [
    'age', 'resting_bp', 'cholesterol', 'max_heart_rate', 'oldpeak',
    'bmi', 'stress_level', 'hr_age_ratio', 'heart_rate_reserve',
    'cholesterol_bp_ratio', 'lifestyle_risk_score', 'comorbidity_score'
]

# Safety: only keep cols that exist
scale_cols = [c for c in scale_cols if c in df.columns]
print(f"Scaling {len(scale_cols)} columns: {scale_cols}")

scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

joblib.dump(scaler, '../models/standard_scaler.pkl')
print("\nScaler saved to '../models/standard_scaler.pkl'")

df[scale_cols].describe().round(3)

## 5. Final Feature Matrix Overview

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

print(f"Feature matrix shape : {X.shape}")
print(f"Target shape         : {y.shape}")
print(f"\nAll features ({len(X.columns)}):")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:>2}. {col}")

In [ ]:
# Correlation heatmap
plt.figure(figsize=(22, 18))
corr_matrix = X.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=False,
    cmap='coolwarm',
    center=0,
    linewidths=0.3,
    cbar_kws={'shrink': 0.6}
)
plt.title('Feature Correlation Heatmap (Post Engineering + Encoding)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Save Processed Dataset

In [ ]:
processed_dir = '../data/processed'
os.makedirs(processed_dir, exist_ok=True)

output_path = f'{processed_dir}/features_engineered.csv'
df.to_csv(output_path, index=False)

print(f"Saved: features_engineered.csv → shape {df.shape}")
print(f"Path : {output_path}")
print("\n✓ Processed dataset saved to '../data/processed/'")

## 7. Summary

| Step | Action | Detail |
|---|---|---|
| Drop ID | Removed `patient_id` | No spurious correlations |
| Feature Engineering | 7 new features | `age_group`, `bp_category`, `hr_age_ratio`, `heart_rate_reserve`, `cholesterol_bp_ratio`, `lifestyle_risk_score`, `comorbidity_score` |
| Ordinal Encoding | 6 columns | `smoking_status`, `alcohol_consumption`, `physical_activity`, `st_slope`, `age_group`, `bp_category` |
| One-Hot Encoding | 4 columns | `gender`, `chest_pain_type`, `resting_ecg`, `thalassemia` · `drop_first=True` |
| StandardScaler | 12 continuous columns | Zero-mean, unit-variance |
| Saved | `features_engineered.csv` + 2 encoder pkls + scaler pkl | Ready for `05_model_training.ipynb` |

➡️ **Next step:** `05_model_training.ipynb` — load `features_engineered.csv`, apply train/test split, handle class imbalance, train and evaluate models.